**Note: Please "join" the competition first. Then, you can mount the dataset to the GPU. Otherwise, the notebook may encounter an error because it cannot access the dataset until you have joined the competition.**

In [1]:
import os
import numpy as np
import astropy.io.fits as pyfits
from scipy.ndimage import gaussian_filter
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [2]:
# Path to training data *** do not change *** 
DATA_DIR = "/bohr/training-lg02/v1/"

DEVICE = "cuda"
MODEL_SAVE_PATH = "model.pth"

BATCH_SIZE = 4
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
SMOOTHING = 3 # parameter for Gaussian Smoothing 

# Example Data Usage

The convergence maps are saved as .fits files in /map folder. The labels are saves as .fits files in /cat folder.

In [3]:
# Show raw convergence map
Z = pyfits.open(os.path.join(DATA_DIR, 'map/1.fits'))[0].data
plt.figure(figsize=(10, 8))
plt.imshow(Z, vmin=-0.1, vmax=0.2, cmap='binary')
plt.show()

In [4]:
# Apply Gaussian Smoothing
Z_smooth = gaussian_filter(Z, sigma=SMOOTHING)
plt.figure(figsize=(10, 8))
plt.imshow(Z_smooth, vmin=-0.1, vmax=0.2, cmap='binary')
plt.show()

In [5]:
# Read halo labels
cat1 = pyfits.open(os.path.join(DATA_DIR, 'cat/1.fits'))[0].data
print(cat1.shape)

cat1 is an ndarray containing 125 rows and 3 columns. Each row represents a labeled halo. The first column is the mass of the halo (irrelevant to this problem), the second and third columns are the y-coordinate and x-coordinate of the halo, respectively, in pixels.

In [6]:
# Visualize halos
cat1_data = np.transpose(cat1)
plt.figure(figsize=(10, 8))
plt.imshow(Z_smooth, vmin=-0.1, vmax=0.2, cmap='binary')
plt.scatter(cat1_data[2], cat1_data[1], facecolors='none', edgecolors='red',s=100)
plt.show()

# Prepare Dataset and Data Loader

A sample dataset for loading the data is provided. You may modify this according to your needs.

In [7]:
class AstroDataset(Dataset):
    def __init__(self, map_paths, cat_paths=None):
        self.map_paths = map_paths
        self.cat_paths = cat_paths
    
    def __len__(self):
        return len(self.map_paths)
    
    def __getitem__(self, idx):
        # Load image with gaussian smoothing
        map_data = gaussian_filter(pyfits.open(self.map_paths[idx])[0].data, sigma=SMOOTHING)
        
        if self.cat_paths is None:
            return torch.FloatTensor(map_data).unsqueeze(0)

        # Load catalog
        cat = pyfits.open(self.cat_paths[idx])[0].data
        target = np.zeros((1024, 1024))

        for y, x in cat[:, 1:3]:
            target[int(y), int(x)] = 1.0

        return torch.FloatTensor(map_data).unsqueeze(0), torch.FloatTensor(target), self.cat_paths[idx]


In [8]:
data_size = len(os.listdir(os.path.join(DATA_DIR, 'map')))

# Ensure correct data and label ordering
dataset = AstroDataset([os.path.join(DATA_DIR, f'map/{i}.fits') for i in range(1, data_size+1)], 
                       [os.path.join(DATA_DIR, f'cat/{i}.fits') for i in range(1, data_size+1)])

# Create data loader
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Define and Train Model

In [9]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Encoder
        self.enc_conv1 = self.conv_block(1, 16)
        self.enc_conv2 = self.conv_block(16, 32)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Bottleneck
        self.bottleneck = self.conv_block(32, 64)
        
        # Decoder
        self.upsample1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec_conv1 = self.conv_block(96, 32)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec_conv2 = self.conv_block(48, 16)
        
        # Output layer
        self.out_conv = nn.Conv2d(16, 1, kernel_size=1)
        
    def conv_block(self, in_channels, out_channels, dropout_rate=0.5):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(negative_slope=0.01, inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(negative_slope=0.01, inplace=True)
        )
    def forward(self, x):
        # Encoder
        enc1 = self.enc_conv1(x)
        enc2 = self.enc_conv2(self.pool(enc1))
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool(enc2))
        
        # Decoder
        up1 = self.upsample1(bottleneck)
        dec1 = self.dec_conv1(torch.cat([up1, enc2], dim=1))
        
        up2 = self.upsample2(dec1)
        dec2 = self.dec_conv2(torch.cat([up2, enc1], dim=1))
        
        # Output layer
        out = self.out_conv(dec2)
        
        return out

In [18]:
# Function for applicating model results
def detect_objects(model, image_path, confidence_threshold, device=DEVICE):
    model.eval()
    with torch.no_grad():
        # Load and preprocess image
        map_data = gaussian_filter(pyfits.open(image_path)[0].data, sigma=SMOOTHING)
        image = torch.FloatTensor(map_data).unsqueeze(0).unsqueeze(0).to(device)
        
        # Get predictions
        output = torch.sigmoid(model(image))
        predictions = output.cpu().squeeze().numpy()
        
        # Convert to coordinates
        coordinates = []
        for y, x in zip(*np.where(predictions > confidence_threshold)):
            confidence = predictions[y, x]
            coordinates.append((x, y, confidence))
            
        return coordinates

# Visualizing model predictions
def visualize_results(model, image_path, label_path, confidence_threshold, device=DEVICE):
    # Get predictions with confidence scores
    results = detect_objects(model, image_path, confidence_threshold, device)
    results = np.array(results).T if results else np.array([[],[],[]])
    
    Z = pyfits.open(image_path)[0].data
    Z_smooth = gaussian_filter(Z, sigma=SMOOTHING)
    
    labels = np.transpose(pyfits.open(label_path)[0].data)
    
    plt.figure(figsize=(10,10))
    plt.imshow(Z_smooth, vmin=-0.1, vmax=0.2, cmap='binary')
    plt.scatter(labels[2], labels[1], facecolors='none', edgecolors='red', s=100, label="True")
    if len(results[0]) > 0:
        # Color the scatter points based on confidence scores
        plt.scatter(results[0], results[1], facecolors='none', edgecolors='green', s=100, label="Predicted")
    plt.legend()
    plt.show()


'''
Metric function for calculating PR-AUC.
This exact function will be used for evaluation
'''
def calculate_precision_recall_curve(predictions, labels):

    print("shape of predictions: ", predictions.shape)

    # Flatten the predictions and get the indices of the sorted predictions
    flat_predictions = predictions.flatten()
    sorted_indices = np.argsort(-flat_predictions)  # Sort in descending order

    precisions = []
    recalls = []

    true_preds = 0
    num_preds = 0
    predicted_labels = 0
    num_labels = sum(len(l) for l in labels)

    labels_within_distance = [[] for _ in range(len(flat_predictions))]

    i = 0
    for image_idx, image_labels in enumerate(labels):
        for y_true, x_true in image_labels:
            for y in range(max(0, int(y_true) - 15), min(1024, int(y_true) + 16)):
                # Calculate the maximum x distance for the current y
                max_x_dist = int((max(0, 15**2 - (y - y_true)**2))**0.5)
                # Calculate the range of x-coordinates
                for x in range(max(0, int(x_true) - max_x_dist), min(1024, int(x_true) + max_x_dist + 1)):
                    coord_idx = image_idx * 1024 * 1024 + y * 1024 + x
                    labels_within_distance[coord_idx].append(i)
            i += 1
    
    label_predicted = [False] * num_labels

    # Iterate over sorted predictions
    for idx in sorted_indices:

        num_preds += 1

        # Determine the image index and the coordinate within the image
        image_idx = idx // (1024 * 1024)
        coord_idx = idx % (1024 * 1024)
        y, x = divmod(coord_idx, 1024)

        if len(labels_within_distance[idx]) > 0:
            true_preds += 1
            for label in labels_within_distance[idx]:
                if label_predicted[label] is False:
                    label_predicted[label] = True
                    predicted_labels += 1

        # Calculate precision and recall
        precision = true_preds / num_preds
        recall = predicted_labels / num_labels

        # Append precision and recall to the lists
        precisions.append(precision)
        recalls.append(recall)

    # Calculate PR-AUC using the trapezoidal rule
    pr_auc = np.trapz(precisions, x=recalls)

    return precisions, recalls, pr_auc

# Evaluate model
def get_pr(model, test_loader, device=DEVICE):
    model.eval()
    for images, _, paths in test_loader:
        with torch.no_grad():
            images = images.to(device)
            outputs = torch.sigmoid(model(images)).cpu().numpy().squeeze(1)
            cat_data = [np.transpose(pyfits.open(path)[0].data) for path in paths]
            labels = [list(zip(cat[1], cat[2])) for cat in cat_data]
        return calculate_precision_recall_curve(outputs, labels)

In [11]:
# Model instance training

model = Model().to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([8000.]).cuda())
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses = []
import tqdm
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    
    for images, targets, _ in tqdm.tqdm(dataloader):
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images).squeeze(1)
        loss = criterion(outputs, targets)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    train_loss = total_loss/len(dataloader)
    train_losses.append(train_loss)
    
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Training Loss: {train_loss:.4f}')

In [19]:
# Plot training curve
plt.figure(figsize=(12, 4))

plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.yscale('log')
plt.title('Training Loss')
plt.grid(True)

plt.show()

# Inference and Testing

In [23]:
visualize_results(model, dataset.map_paths[0], dataset.cat_paths[0], confidence_threshold=0.5)

In [21]:
# Calculate PR-AUC
precisions, recalls, pr_auc = get_pr(model, dataloader)
print(f"PR-AUC Score: {pr_auc:.4f}")

# Plot PR curve
plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve (AUC = {pr_auc:.4f})')
plt.grid(True)
plt.show()

# Submission 
In your submitted code, the environment variable ```DATA_PATH``` will be assigned a value to the testing dataset. This dataset is inaccessible by contestants. You can modify this code based on how your model inferencing works, but do not change the shape or format of the submitted files. While the full training code is provided in the baseline, you should only submit the inference section of your code. You should submit your best model parameters by creating a dataset on Bohrium including your model weights, and attaching the dataset to your submission. Your inference code should read your model weights from the dataset (from the /bohr path) and read test data from the DATA_PATH environment variable. The following submission code is for your reference. **See the problem description for more details on submission.**

IMPORTANT: You are required to keep the full source code (.ipynb) that resulted in your best model. The competition committee reserves the right to cancel your scores if you fail to provide code that reproduces your best model when requested. You are encouraged to employ practices such as setting constant random seeds to ensure reproducibility.

In [ ]:
import zipfile

if os.environ.get('DATA_PATH'):
    # Submit testA for public leaderboard 
    DATA_PATH = os.environ.get('DATA_PATH') + "/"
    TEST_DATA_DIR = DATA_PATH + 'halo_testA'
    test_size = len(os.listdir(os.path.join(TEST_DATA_DIR, 'map')))
    dataset = AstroDataset([os.path.join(TEST_DATA_DIR, f'map/{i}.fits') for i in range(1, test_size+1)])
    loader = DataLoader(dataset, batch_size=test_size, shuffle=False)
    model.eval()
    for images in loader:
        with torch.no_grad():
            outputs = torch.sigmoid(model(images.to(DEVICE))).cpu().numpy().squeeze(1)
        np.save('submissionsA.npy', outputs)
    # Submit testB for private leaderboard
    TEST_DATA_DIR = DATA_PATH + 'halo_testB'
    test_size = len(os.listdir(os.path.join(TEST_DATA_DIR, 'map')))
    dataset = AstroDataset([os.path.join(TEST_DATA_DIR, f'map/{i}.fits') for i in range(1, test_size+1)])
    loader = DataLoader(dataset, batch_size=test_size, shuffle=False)
    model.eval()
    for images in loader:
        with torch.no_grad():
            outputs = torch.sigmoid(model(images.to(DEVICE))).cpu().numpy().squeeze(1)
        np.save('submissionsB.npy', outputs)
    
    # The final submission will be a zip file containing the your model outputs for both testing sets
    with zipfile.ZipFile('submission.zip', 'w') as zipf:
        zipf.write('submissionsA.npy')
        zipf.write('submissionsB.npy')